# 第1日 ノートブック02：デジタル画像を理解する

# Day 1 Notebook 02: Understanding Digital Images

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. デジタル画像が「数値の表」であることを説明できる | 1. Explain that a digital image is a table of numbers |
| 2. 画素数・解像度・幅・高さの関係を説明できる | 2. Relate pixel count, resolution, width and height |
| 3. RGB とグレースケールの違いを説明できる | 3. Distinguish RGB from greyscale |
| 4. NumPy 配列で画像を作り、書き換えられる | 4. Create and modify an image as a NumPy array |
| 5. 解像度を下げると情報が失われることを、実際に確認できる | 5. Demonstrate the information lost when resolution is reduced |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 90 分 / 90 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日ノートブック01 を終えていること。 / You have finished Day 1 Notebook 01. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 画像を作る・書き換える演習の結果 | Results of the image creation/modification exercises |
| 低解像度化で失われる情報についての考察 | Your note on the information lost at low resolution |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師の説明：画像＝数字の表 | Instructor: an image is a table of numbers |
| 09:00–10:30 | Part 1〜Part 3（画素・色・配列） | Part 1-Part 3 (pixels, colour, arrays) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 6（座標・拡大縮小・予想演習） | Part 4-Part 6 (coordinates, resizing, prediction) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：午前の振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・理解度チェック・提出物 | Exercises, questions, submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

#### 日本語：図の日本語フォントを設定する
**日本語：** Matplotlib の既定フォントは日本語を表示できません。そのままだと図のタイトルが `□□□` になります。次のセルで、日本語を表示できるフォントを探して設定します。Mac や Linux では自動で別のフォントが選ばれます。
#### English: Configure Japanese text in figures
**English:** Matplotlib's default font cannot draw Japanese characters, so figure titles would appear as `□□□`. The next cell looks for a Japanese font and selects it. On macOS or Linux a different font will be chosen automatically.

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def show_png(relative_parts, caption=""):
    """素材フォルダの PNG を表示する / display a PNG from the assets folder."""
    path = ASSETS.joinpath(*relative_parts)
    if not path.is_file():
        print("画像がありません / file not found:", path)
        print("assets\\scripts\\generate_training_assets.py を実行してください。")
        return
    image = Image.open(path)
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.axis("off")
    if caption:
        plt.title(caption)
    plt.show()
    print("size (width, height) =", image.size, " mode =", image.mode)

---

## Part 1　画素とは / What is a pixel

#### 日本語：画素（ピクセル）
**日本語：** デジタル画像は、小さな四角が **縦横に並んだ** ものです。
この 1 つの四角を **画素（ピクセル、pixel）** と呼びます。

各画素は **1 つの数値**（白黒の場合）を持ちます。

- `0` = 黒
- `255` = 白
- 中間の値 = 灰色

なぜ 0〜255 かというと、1 画素を 8 ビット（2 の 8 乗 = 256 通り）で表すからです。
**0 から 255 までの 256 段階** しか無い、ということを覚えておいてください。

下の図を見てください。左が 8×8 画素の画像、右がそれを拡大して数値を表示したものです。
#### English: Pixel
**English:** A digital image is a grid of small squares called **pixels**.

In a greyscale image each pixel holds **one number**:

- `0` = black
- `255` = white
- anything between = grey

Why 0-255? Because one pixel is stored in 8 bits (2^8 = 256 values). Remember: there are only
**256 brightness levels**.

Look at the figure below: on the left an 8x8 image, on the right the same image enlarged with the
numbers shown.

In [ ]:
show_png(["diagrams", "diagram_pixel_grid.png"], "8x8 pixels and their numbers")

#### 日本語：数値と明るさの対応
**日本語：** 下の図は、0〜255 の値が実際にどの明るさに見えるかを示しています。
数値が大きいほど白に近づきます。

**English:** The next figure shows how the values 0-255 actually look. Larger numbers are closer
to white.
#### English: The relationship between numbers and brightness

In [ ]:
show_png(["diagrams", "diagram_grayscale_values.png"], "0 = black, 255 = white")

---

## Part 2　幅・高さ・解像度 / Width, height and resolution

#### 日本語：3 つの言葉の違い
**日本語：** よく混同される 3 つの言葉を整理します。

| 言葉 | 意味 | 例 |
|---|---|---|
| 幅（width） | 横方向の画素数 | 640 |
| 高さ（height） | 縦方向の画素数 | 480 |
| 画素数（pixels） | 幅 × 高さ | 640 × 480 = 307,200 |
| 解像度（resolution） | 画素がどれだけ細かいか。ふつう画素数で表す | 30 万画素 |

**日本語：** 大事なのは、**幅と高さを掛けた数が、画像に含まれる数値の個数** だということです。

**English:** Three easily confused terms:

| Term | Meaning | Example |
|---|---|---|
| width | number of pixels horizontally | 640 |
| height | number of pixels vertically | 480 |
| pixel count | width x height | 640 x 480 = 307,200 |
| resolution | how fine the pixels are; usually expressed as pixel count | 0.3 megapixels |

The key idea: **width x height is the number of numbers stored in the image**.
#### English: Three related terms

In [ ]:
import numpy as np

# 8 x 8 の画像 = 64 個の数値 / an 8x8 image = 64 numbers
small = np.full((8, 8), 200, dtype=np.uint8)

print("small.shape =", small.shape)
print("  -> 高さ (height) =", small.shape[0])
print("  -> 幅   (width)  =", small.shape[1])
print("  -> 数値の個数    =", small.size)

# 実際のカメラに近い大きさ / a size closer to a real camera
print()
print("640 x 480 の場合 / for 640 x 480 :", 640 * 480, "個の数値")
print("1920 x 1080 の場合 / for 1920 x 1080:", 1920 * 1080, "個の数値")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
small.shape = (8, 8)
  -> 高さ (height) = 8
  -> 幅   (width)  = 8
  -> 数値の個数    = 64

640 x 480 の場合 / for 640 x 480 : 307200 個の数値
1920 x 1080 の場合 / for 1920 x 1080: 2073600 個の数値
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
small.shape = (8, 8)
  -> height = 8
  -> width  = 8
  -> number of values = 64

for 640 x 480 : 307200 values
for 1920 x 1080: 2073600 values
```

#### 日本語：解像度と、見える欠陥の大きさ
**日本語：** ここが製造検査で決定的に重要です。

シートの幅 1,300 mm を、幅 1,300 画素のカメラで撮るとします。すると **1 画素 = 1 mm** です。
0.3 mm の欠陥は、**1 画素より小さい** ので、写りません（または 1 画素の薄い影になるだけ）。

幅 13,000 画素で撮れば 1 画素 = 0.1 mm になり、0.3 mm の欠陥は **3 画素** で写ります。

> **日本語：** 「AI の精度が悪い」と言う前に、**その欠陥はそもそも画像に写っているか** を
> 確認する必要があります。

**English:** This is decisive for inspection.

Suppose the 1,300 mm sheet width is imaged with a 1,300-pixel-wide camera: **1 pixel = 1 mm**.
A 0.3 mm defect is **smaller than one pixel**, so it cannot be seen (it becomes a faint shadow).

With 13,000 pixels across, 1 pixel = 0.1 mm and the 0.3 mm defect covers **three pixels**.

Before claiming "the AI is inaccurate", check **whether the defect is present in the image at all**.
#### English: Resolution and the smallest visible defect

---

## Part 3　RGB とグレースケール / RGB and greyscale

#### 日本語：カラー画像は 3 枚の表
**日本語：** カラー画像は、**赤・緑・青** の 3 つの数値で 1 画素を表します。
つまり 1 画素あたり 3 つの数値です。

```text
画素 (x=3, y=2) = 赤 210, 緑 205, 青 190
```

3 つが同じ値なら灰色になります。だから **グレースケール（白黒）** は 1 画素 1 数値で済みます。

| 種類 | 1 画素あたりの数値の個数 | 配列の形 |
|---|---|---|
| グレースケール | 1 | (高さ, 幅) |
| RGB カラー | 3 | (高さ, 幅, 3) |

**日本語：** NumPy では、カラー画像は `shape = (高さ, 幅, 3)` の配列になります。
`3` の部分が「赤・緑・青」です。**この順番（RGB）を覚えてください。**
OpenCV では **BGR**（青・緑・赤）の順になるので注意が必要です（第 2 日で扱います）。

**English:** A colour image stores **red, green and blue** for every pixel — three numbers per pixel.

```text
pixel (x=3, y=2) = red 210, green 205, blue 190
```

If all three values are equal you get grey, which is why **greyscale** needs only one number.

| Kind | Numbers per pixel | Array shape |
|---|---|---|
| greyscale | 1 | (height, width) |
| RGB colour | 3 | (height, width, 3) |

In NumPy a colour image has `shape = (height, width, 3)`. The `3` is red, green, blue.
Remember that order. OpenCV uses **BGR** instead — Day 2 covers this.
#### English: A colour image is three tables

In [ ]:
# 3 x 4 のカラー画像を作る / build a small 3x4 colour image
colour = np.zeros((3, 4, 3), dtype=np.uint8)

colour[0, 0] = [255, 0, 0]      # 赤 / red
colour[0, 1] = [0, 255, 0]      # 緑 / green
colour[0, 2] = [0, 0, 255]      # 青 / blue
colour[0, 3] = [255, 255, 255]  # 白 / white
colour[1, :] = [210, 205, 190]  # PVC シートのような、少し黄色みのある白 / PVC-like off-white
colour[2, :] = [40, 40, 40]     # 濃い灰色 / dark grey

print("shape:", colour.shape)
print()
print("画素 (0,0) の値 / pixel (0,0):", colour[0, 0])
print("画素 (1,0) の値 / pixel (1,0):", colour[1, 0])

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
shape: (3, 4, 3)

画素 (0,0) の値 / pixel (0,0): [255   0   0]
画素 (1,0) の値 / pixel (1,0): [210 205 190]
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
shape: (3, 4, 3)

pixel (0,0): [255   0   0]
pixel (1,0): [210 205 190]
```

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.imshow(colour, interpolation="nearest")
plt.title("3 x 4 colour image / 3×4 のカラー画像")
plt.xticks(range(4))
plt.yticks(range(3))
plt.show()

#### 日本語：グレースケールへの変換
**日本語：** RGB からグレースケールへの変換は、3 つの値の **重み付き平均** です。
人間の目は緑に敏感なので、緑の重みが大きくなっています。

```text
グレー = 0.299 × 赤 + 0.587 × 緑 + 0.114 × 青
```

**日本語：** この式を暗記する必要はありません。「3 つの数値を 1 つにまとめている」
と理解すれば十分です。

**English:** The RGB-to-greyscale conversion is a **weighted average**. Human eyes are most
sensitive to green, hence the larger weight.

```text
grey = 0.299 * red + 0.587 * green + 0.114 * blue
```

You do not need to memorise it; "three numbers are combined into one" is enough.
#### English: Converting to greyscale

In [ ]:
# 自分で計算する / compute it manually
red, green, blue = colour[1, 0]
manual_grey = 0.299 * red + 0.587 * green + 0.114 * blue
print("手計算 / manual        :", round(float(manual_grey), 1))

# 重み付き平均を配列全体に適用 / apply the weighted average to the whole array
weights = np.array([0.299, 0.587, 0.114])
grey_array = (colour.astype(np.float64) * weights).sum(axis=2)
print("配列全体 / whole array :", np.round(grey_array, 1))
print()
print("グレースケールの shape:", grey_array.shape, "（3 次元 -> 2 次元）")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
手計算 / manual        : 204.8
配列全体 / whole array : [[ 76.2  149.7  29.1 255. ]
 [204.8 204.8 204.8 204.8]
 [ 40.   40.   40.   40. ]]
グレースケールの shape: (3, 4) （3 次元 -> 2 次元）
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
manual        : 204.8
whole array   : [[ 76.2  149.7  29.1 255. ]
 [204.8 204.8 204.8 204.8]
 [ 40.   40.   40.   40. ]]
greyscale shape: (3, 4)   (3-D -> 2-D)
```

---

## Part 4　画像の座標 / Image coordinates

#### 日本語：原点は左上、y は下向き
**日本語：** 数学のグラフとは **逆** です。注意してください。

- 原点 `(0, 0)` は **左上** の画素
- `x` は **右** に増える
- `y` は **下** に増える

NumPy の配列も同じです。`image[y, x]` の順に書きます。**`x` と `y` の順番が逆** なので、
最初は必ず間違えます。落ち着いて確認してください。

**English:** This is the **opposite** of a maths graph:

- the origin `(0, 0)` is the **top-left** pixel
- `x` increases to the **right**
- `y` increases **downward**

NumPy arrays follow the same convention, and we index as `image[y, x]` — **row first, column
second**. Everyone gets this backwards at first; check carefully.
#### English: The origin is top-left, and y increases downward

In [ ]:
show_png(["diagrams", "diagram_coordinate_system.png"], "x to the right, y downward")

In [ ]:
# 座標の確認 / verifying the coordinate convention
canvas = np.zeros((5, 7), dtype=np.uint8)     # 高さ 5、幅 7

canvas[0, 0] = 255      # 左上 / top-left
canvas[0, 6] = 200      # 右上 / top-right
canvas[4, 0] = 150      # 左下 / bottom-left
canvas[2, 3] = 100      # 中央 / centre

print("canvas.shape =", canvas.shape, "  (高さ, 幅) = (height, width)")
print()
for row in canvas:
    print(" ".join(f"{value:3d}" for value in row))

print()
print("canvas[0, 6] は右上の画素 / canvas[0, 6] is the top-right pixel")
print("canvas[4, 0] は左下の画素 / canvas[4, 0] is the bottom-left pixel")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
canvas.shape = (5, 7)   (高さ, 幅) = (height, width)

255   0   0   0   0   0 200
  0   0   0   0   0   0   0
  0   0   0 100   0   0   0
  0   0   0   0   0   0   0
150   0   0   0   0   0   0
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
canvas.shape = (5, 7)   (height, width)

255   0   0   0   0   0 200
  0   0   0   0   0   0   0
  0   0   0 100   0   0   0
  0   0   0   0   0   0   0
150   0   0   0   0   0   0
```

---

## Part 5　画像を作って書き換える / Creating and modifying images

#### 日本語：シートらしい画像を作る
**日本語：** ここからは、実際に「PVC シートらしい」画像を作って操作します。
本物の工場画像ではありませんが、明るさのムラやノイズの考え方を練習できます。

手順は 3 つです。

1. 全体を明るい灰色（`210`）で埋める
2. ゆっくり変化するムラを加える
3. 細かいノイズを加える
#### English: Build a sheet-like image
**English:** From here on we build a "PVC-sheet-like" image and operate on it. It is not a real
factory image, but it lets you practise brightness variation and noise.

Three steps:

1. fill everything with light grey (`210`)
2. add a slowly varying brightness gradient
3. add fine noise

In [ ]:
rng = np.random.default_rng(seed=5)
height, width = 120, 200

# 1. 明るい灰色で埋める / fill with light grey
sheet = np.full((height, width), 210.0)

# 2. ゆっくりしたムラ / slowly varying shading
yy, xx = np.mgrid[0:height, 0:width]
shading = 6.0 * np.sin(xx / 40.0) * np.cos(yy / 30.0)
sheet = sheet + shading

# 3. 細かいノイズ / fine noise
sheet = sheet + rng.normal(0.0, 3.0, size=(height, width))

sheet = np.clip(sheet, 0, 255).astype(np.uint8)

print("shape:", sheet.shape, " dtype:", sheet.dtype)
print("min / mean / max:", sheet.min(), round(float(sheet.mean()), 1), sheet.max())

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
shape: (120, 200)  dtype: uint8
min / mean / max: 192 209.4 225
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
shape: (120, 200)  dtype: uint8
min / mean / max: 192 209.4 225
```

In [ ]:
plt.figure(figsize=(9, 4))
plt.imshow(sheet, cmap="gray", vmin=0, vmax=255)
plt.title("synthetic PVC-like sheet / 合成 PVC シート風の画像")
plt.colorbar(label="brightness")
plt.show()

#### 日本語：画素を書き換えて欠陥を作る
**日本語：** 画素の値を書き換えると、その部分が暗くなります。これが「欠陥」の最小の例です。

下のセルは 6×6 画素の黒い四角を作ります。`sheet[60:66, 100:106]` は
「y が 60〜65、x が 100〜105」という意味です。
#### English: Modify pixels to create a defect
**English:** Overwriting pixel values makes that region darker — the simplest possible "defect".

The next cell creates a 6x6 dark square. `sheet[60:66, 100:106]` means
"y from 60 to 65, x from 100 to 105".

In [ ]:
defective = sheet.copy()        # 元を壊さないためにコピー / copy so the original is preserved
defective[60:66, 100:106] = 40

print("元の画像の値 (60, 100):", sheet[60, 100])
print("書き換えた値 (60, 100):", defective[60, 100])
print("暗くなった画素数      :", int((defective < 100).sum()))

plt.figure(figsize=(9, 4))
plt.imshow(defective, cmap="gray", vmin=0, vmax=255)
plt.title("with a 6x6 dark square / 6×6 の黒い四角を追加")
plt.colorbar(label="brightness")
plt.show()

---

## Part 6　拡大・縮小で失われる情報 / Information lost when resizing

#### 日本語：縮小すると何が起きるか
**日本語：** 画像を縮小すると、**画素が減る** ので、情報が失われます。
具体的には、隣り合う画素が 1 つにまとめられます。

小さな欠陥は、縮小すると消えてしまうことがあります。これは製造検査で非常に重要です。

**English:** Shrinking an image **reduces the number of pixels**, so information is lost:
neighbouring pixels are merged into one. Small defects can disappear entirely.
That matters enormously in inspection.
#### English: What happens when you shrink an image

In [ ]:
from PIL import Image

def resize_array(array, new_width, new_height):
    """NumPy 配列を Pillow で拡大縮小する / resize a NumPy array with Pillow."""
    pil_image = Image.fromarray(array)
    resized = pil_image.resize((new_width, new_height), Image.BILINEAR)
    return np.asarray(resized)


sizes = [(200, 120), (100, 60), (50, 30), (25, 15)]

fig, axes = plt.subplots(1, len(sizes), figsize=(15, 3.8))
for ax, (w, h) in zip(axes, sizes):
    small_view = resize_array(defective, w, h)
    ax.imshow(small_view, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{w} x {h}\n{small_view.size} pixels")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("same image at four resolutions / 同じ画像を 4 つの解像度で")
plt.tight_layout()
plt.show()

#### 日本語：欠陥はいつ消えるか
**日本語：** 下のセルは、解像度ごとに「一番暗い画素」を調べます。
欠陥が 1 画素に満たなくなると、値が急に「普通の明るさ」に近づきます。

**English:** The next cell reports the darkest pixel at each resolution. Once the defect is smaller
than a pixel, that value jumps back towards normal brightness.
#### English: When does the defect vanish

In [ ]:
print(f"{'解像度 / size':>14}  {'最小値 / min':>12}  {'欠陥は残っているか / defect visible':>24}")
print("-" * 60)
for w, h in sizes:
    view = resize_array(defective, w, h)
    darkest = int(view.min())
    visible = "YES / はい" if darkest < 150 else "NO / いいえ"
    print(f"{w:>6} x {h:<5}  {darkest:>12}  {visible:>24}")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
解像度 / size     最小値 / min  欠陥は残っているか / defect visible
------------------------------------------------------------
   200 x 120              40                  YES / はい
   100 x 60               40                  YES / はい
    50 x 30               79                  YES / はい
    25 x 15              171                  NO / いいえ
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
size             min      defect visible
--------------------------------------------
   200 x 120         40      YES
   100 x 60          40      YES
    50 x 30          79      YES
    25 x 15         171      NO
```

---

## Part 7　予想してから実行する / Predict before you run

#### 日本語：結果を予想してください
**日本語：** 下のセルは、`predict_me` という画像を作ります。**実行する前に**、
ノートに次の 3 つを書いてください。

1. 画像は何色に見えるか
2. どこに濃い部分ができるか
3. 全体の平均の明るさは、いくつくらいか

書いたら実行して、予想と比べてください。

**English:** The next cell builds an image called `predict_me`. **Before running it**, write down:

1. what colour it will look like
2. where the dark region will be
3. roughly what the average brightness will be

Then run it and compare with your prediction.
#### English: Predict the result

In [ ]:
predict_me = np.full((60, 60, 3), 200, dtype=np.uint8)   # 明るい灰色（RGB すべて 200）
predict_me[10:25, 40:55] = [60, 60, 60]                  # 一部を濃い灰色に
predict_me[:, :20, 0] = 230                               # 左 20 列は赤成分を強く

print("shape:", predict_me.shape)
print("mean per channel (R, G, B):", np.round(predict_me.reshape(-1, 3).mean(axis=0), 1))

plt.figure(figsize=(4, 4))
plt.imshow(predict_me, interpolation="nearest")
plt.title("predict_me")
plt.show()

#### 日本語：なぜそうなったか
**日本語：** 左側が赤っぽく見えたのは、赤の値だけを `230` に上げたからです。
右側の濃い部分は、3 つの値をすべて `60` にしたので灰色です。

**日本語：** この「予想してから実行する」練習は、これからの 5 日間ずっと有効です。
**予想が外れたときが、一番学べる瞬間** です。

**English:** The left part looks reddish because only the red channel was raised to `230`. The
darker square on the right is grey because all three channels were set to `60`.

Predict-before-you-run stays useful for the whole five days. **The moment your prediction is
wrong is the moment you learn the most.**
#### English: Why it looks like that

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：自分の画像を作る
**日本語：** 100×100 のグレースケール画像を作り、中央 20×20 の画素を `30` にしてください。
さらに `plt.imshow()` で表示してください。

**English:** Build a 100x100 greyscale image and set the central 20x20 pixels to `30`, then display
it with `plt.imshow()`.
#### English: Exercise 1 (basic): make your own image
Done when the displayed image shows a dark square exactly in the centre.

In [ ]:
# 演習 1 / Exercise 1
import numpy as np
import matplotlib.pyplot as plt

my_image = np.full((100, 100), 220, dtype=np.uint8)
# 中央 20x20 を 30 にする / set the central 20x20 to 30
# ヒント: y は 40〜59、x は 40〜59 / hint: y 40-59, x 40-59


plt.figure(figsize=(4, 4))
plt.imshow(my_image, cmap="gray", vmin=0, vmax=255)
plt.title("my image")
plt.show()

#### 日本語：演習 2（中級）：色を変える
**日本語：** 演習 1 の画像を **カラー** にして、中央の四角を「赤っぽい灰色」にしてください。
そのあと、赤成分だけを残して表示し（緑と青を 0 にする）、何が見えるか確認してください。

**English:** Turn your Exercise 1 image into a **colour** image and make the centre square a
reddish grey. Then display only the red channel (set green and blue to 0) and describe what you see.
#### English: Exercise 2 (intermediate): change colours
Done when you can explain why the image looks like a bright shape on black.

In [ ]:
# 演習 2 / Exercise 2
# ヒント / hint: np.zeros((100, 100, 3), dtype=np.uint8)


# 赤成分だけ表示 / show only red
# red_only = ...

#### 日本語：演習 3（発展）：解像度と欠陥の関係を説明する
**日本語：** Part 6 の結果を使い、「幅 1,300 mm のシートで、0.3 mm の欠陥を見つけるには、
横方向に何画素必要か」を計算してください。理由も書いてください。

ヒント：1 つの欠陥を最低 3 画素で写したいと仮定します。

**English:** Using the Part 6 results, calculate how many pixels across are needed to detect a
0.3 mm defect on a 1,300 mm wide sheet. Explain your reasoning.
Hint: assume the defect must cover at least 3 pixels.
#### English: Exercise 3 (challenge): relate resolution to detectability
Done when the number of pixels and the assumption behind it are both written down.

In [ ]:
# 演習 3 / Exercise 3
sheet_width_mm = 1300.0
defect_mm = 0.3
pixels_per_defect = 3

# 計算 / calculation:
# required_pixels = ...

# 理由 / reasoning:
#

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. 640×480 のカラー画像には、数値がいくつ入っていますか。計算式も書いてください。
2. 画像の座標で、`(0, 0)` はどこですか。数学のグラフとどう違いますか。
3. RGB の 3 つの値がすべて同じ 120 のとき、その画素は何色ですか。
4. 解像度を下げると、小さな欠陥はどうなりますか。理由も書いてください。
5. 「画素を書き換える」ことは、製造現場では何に相当しますか。

**English:**

1. How many numbers are stored in a 640x480 colour image? Show the calculation.
2. Where is `(0, 0)` in image coordinates, and how does that differ from a maths graph?
3. A pixel has R=G=B=120. What colour is it?
4. What happens to a small defect when you lower the resolution, and why?
5. In a factory, what does "overwriting a pixel" correspond to?
#### English: Understanding questions
Model answers: `docs/instructor/day01_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `FileNotFoundError` で画像が読めない | `FileNotFoundError` when opening an image | 素材が未生成です。PowerShell で `python assets\scripts\generate_training_assets.py` を 1 回実行してください。 | Assets were never generated. Run `python assets\scripts\generate_training_assets.py` once. |
| `TypeError: Invalid shape ... for image data` | `TypeError: Invalid shape ... for image data` | `imshow` に渡す配列の次元が違います。グレーは 2 次元、カラーは 3 次元です。 | The array passed to `imshow` has the wrong number of dimensions (2 for grey, 3 for colour). |
| 画像が真っ黒／真っ白になる | The image looks entirely black or white | `vmin` と `vmax` を指定してください（`vmin=0, vmax=255`）。指定しないと自動調整されます。 | Specify `vmin` and `vmax`; without them Matplotlib rescales automatically. |
| `IndexError: index 200 is out of bounds` | `IndexError: index 200 is out of bounds` | 座標が画像の外です。`shape` を確認してください（`shape[0]` が高さ、`shape[1]` が幅）。 | The coordinate is outside the image. Check `shape` (height first, then width). |
| 色がおかしい（赤と青が入れ替わって見える） | Colours look swapped (red/blue) | OpenCV は BGR 順です。`cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` で変換してください（第 2 日）。 | OpenCV uses BGR. Convert with `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` (Day 2). |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 7 を読んだ
      - [ ] Read Parts 1-7
- [ ] すべてのコードセルを実行した
      - [ ] Ran all code cells
- [ ] 予想を書いてから Part 7 を実行した
      - [ ] Wrote a prediction before running Part 7
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 解像度と欠陥サイズの関係を説明できる
      - [ ] Can explain resolution versus defect size
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [NumPy 配列の基礎](https://numpy.org/doc/stable/user/basics.creation.html)
- [Matplotlib imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)
- [Pillow の resize](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.resize)
- [デジタル画像の基礎（OpenCV チュートリアル）](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `Image.resize` の `Image.BILINEAR` を `Image.NEAREST` に変えて、Part 6 の表を作り直してください。結果はどう変わりましたか？ その理由は？

**English:** Change `Image.BILINEAR` to `Image.NEAREST` in Part 6 and rebuild the table. What changed, and why?